In [1]:
# Import required libraries
from openai import OpenAI

# Initialize the OpenAI client pointing to vLLM server
client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="dummy-key"  # vLLM doesn't require a real API key
)

# Send a test prompt
response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {"role": "user", "content": "What is the capital of France?"}
    ],
    temperature=0.7,
    max_tokens=10000
)

# Print the response
print("Response from vLLM:")
print(response.choices[0].message.content)

Response from vLLM:
The capital of France is **Paris**.


## Setup Complete! 

The semantic cache has been updated to use **nomic-embed-text-v1.5** (768 dimensions) for local embedding generation.

**Changes made:**
1. Replaced OpenAI API embeddings with local `sentence-transformers` library
2. Using `nomic-ai/nomic-embed-text-v1.5` model (768-dimensional embeddings)
3. Updated Qdrant collection to use 768 dimensions
4. Fixed point ID format to use UUID instead of raw hash strings

**To use the cache, restart the kernel and run the cells in order.**


In [3]:
# Import the semantic cache components
from semantic_cache import SemanticCache
from cached_llm_client import CachedLLMClient

# Initialize the semantic cache with local embeddings
cache = SemanticCache(
    qdrant_host="localhost",
    qdrant_port=6333,
    postgres_config={
        "host": "localhost",
        "port": "5432",
        "database": "devdb",
        "user": "postgres",
        "password": "postgres"
    },
    similarity_threshold=0.95,  # 95% similarity required for cache hit
    embedding_model="nomic-ai/nomic-embed-text-v1.5"
)

# Wrap the existing vLLM client with caching
cached_client = CachedLLMClient(
    llm_client=client,
    cache=cache,
    enable_cache=True,
    verbose=True  # Show cache hits/misses
)


/home/tansanrao/work/6204-project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading embedding model: nomic-ai/nomic-embed-text-v1.5


<All keys matched successfully>


Using existing Qdrant collection: llm_cache with dimension 768
PostgreSQL table initialized


In [2]:
# Manually delete and recreate the Qdrant collection with correct dimensions
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams

qdrant_client = QdrantClient(host="localhost", port=6333)

# Delete the old collection
try:
    qdrant_client.delete_collection(collection_name="llm_cache")
    print("Deleted old collection 'llm_cache'")
except Exception as e:
    print(f"Could not delete collection: {e}")

# Create new collection with 768 dimensions (nomic-embed-text-v1.5)
qdrant_client.create_collection(
    collection_name="llm_cache",
    vectors_config=VectorParams(size=768, distance=Distance.COSINE)
)
print("Created new collection 'llm_cache' with 768 dimensions")

# Verify the collection
collection_info = qdrant_client.get_collection("llm_cache")
print(f"Collection info: {collection_info}")


Deleted old collection 'llm_cache'
Created new collection 'llm_cache' with 768 dimensions
Collection info: status=<CollectionStatus.GREEN: 'green'> optimizer_status=<OptimizersStatusOneOf.OK: 'ok'> vectors_count=None indexed_vectors_count=0 points_count=0 segments_count=8 config=CollectionConfig(params=CollectionParams(vectors=VectorParams(size=768, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None, multivector_config=None), shard_number=1, sharding_method=None, replication_factor=1, write_consistency_factor=1, read_fan_out_factor=None, on_disk_payload=True, sparse_vectors=None), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=False, payload_m=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size=None, memmap_threshold=None, indexing_threshold=10000, flush_interval_sec=5, max_optimizatio

In [4]:
# Test 1: First call - should be a cache MISS
print("=" * 60)
print("TEST 1: First call - should query the LLM")
print("=" * 60)

response = cached_client.chat_completions_create(
    model="openai/gpt-oss-20b",
    messages=[
        {"role": "user", "content": "What is the capital of France?"}
    ],
    temperature=0.7,
    max_tokens=10000
)

print(f"\nResponse: {response.choices[0].message.content}")
print(f"Cached: {response.cached}")

TEST 1: First call - should query the LLM
✗ Cache MISS - calling LLM
✗ Cache MISS - calling LLM

Response: The capital of France is **Paris**.
Cached: False

Response: The capital of France is **Paris**.
Cached: False


In [5]:
# Test 2: Exact same question - should be a cache HIT
print("=" * 60)
print("TEST 2: Same question - should hit cache")
print("=" * 60)

response = cached_client.chat_completions_create(
    model="openai/gpt-oss-20b",
    messages=[
        {"role": "user", "content": "What is the capital of France?"}
    ],
    temperature=0.7,
    max_tokens=10000
)

print(f"\nResponse: {response.choices[0].message.content}")
print(f"Cached: {response.cached}")
print(f"Similarity Score: {response.similarity_score:.4f}")

TEST 2: Same question - should hit cache
✓ Cache HIT (similarity: 1.0000)

Response: The capital of France is **Paris**.
Cached: True
Similarity Score: 1.0000


In [6]:
# Test 3: Semantically similar question - should be a cache HIT
print("=" * 60)
print("TEST 3: Semantically similar question - should hit cache")
print("=" * 60)

response = cached_client.chat_completions_create(
    model="openai/gpt-oss-20b",
    messages=[
        {"role": "user", "content": "What's the capital city of France?"}
    ],
    temperature=0.7,
    max_tokens=10000
)

print(f"\nResponse: {response.choices[0].message.content}")
print(f"Cached: {response.cached}")
if hasattr(response, 'similarity_score'):
    print(f"Similarity Score: {response.similarity_score:.4f}")

TEST 3: Semantically similar question - should hit cache
✓ Cache HIT (similarity: 0.9855)

Response: The capital of France is **Paris**.
Cached: True
Similarity Score: 0.9855


In [7]:
# Test 4: Different question - should be a cache MISS
print("=" * 60)
print("TEST 4: Different question - should query LLM")
print("=" * 60)

response = cached_client.chat_completions_create(
    model="openai/gpt-oss-20b",
    messages=[
        {"role": "user", "content": "What is the capital of Germany?"}
    ],
    temperature=0.7,
    max_tokens=10000
)

print(f"\nResponse: {response.choices[0].message.content}")
print(f"Cached: {response.cached}")

TEST 4: Different question - should query LLM
✗ Cache MISS - calling LLM

Response: The capital of Germany is **Berlin**.
Cached: False

Response: The capital of Germany is **Berlin**.
Cached: False


In [8]:
# View cache statistics
print("=" * 60)
print("CACHE STATISTICS")
print("=" * 60)

stats = cached_client.get_stats()

print("\nClient Statistics:")
print(f"  Total Requests: {stats['client_stats']['total_requests']}")
print(f"  Cache Hits: {stats['client_stats']['cache_hits']}")
print(f"  Cache Misses: {stats['client_stats']['cache_misses']}")
print(f"  Hit Rate: {stats['client_stats']['hit_rate']:.1%}")

print("\nCache Database Statistics:")
print(f"  Total Entries: {stats['cache_stats']['total_entries']}")
print(f"  Total Accesses: {stats['cache_stats']['total_accesses']}")
print(f"  Avg Accesses per Entry: {stats['cache_stats']['avg_accesses_per_entry']:.2f}")
print(f"  Qdrant Points: {stats['cache_stats']['qdrant_points']}")

CACHE STATISTICS

Client Statistics:
  Total Requests: 4
  Cache Hits: 2
  Cache Misses: 2
  Hit Rate: 50.0%

Cache Database Statistics:
  Total Entries: 2
  Total Accesses: 4
  Avg Accesses per Entry: 2.00
  Qdrant Points: 2
